# Week 4 lecture walkthrough: persistent homology as homology with memory

This is the live worked example, built around a small class that dies under inclusion, followed by a noisy-circle barcode. It follows the conceptual argument of the slides through prediction, reveal and interpretation. It is not intended as a line-by-line answer key to the participant practical.

**Resource boundary.** The [reference notes](index.qmd) define induced maps, modules and interval summaries. The [slides](slides.qmd) carry the birth-and-death argument. This notebook completes the hand calculation before calling a library. The [participant practical](lab.ipynb) asks students to track the same events and justify their interpretation.

**Lecture map.** First show why an old cycle can map to zero after a face enters. Then show why component generators can merge. Only after the maps are visible should the barcode compress the story into intervals.

We move through

$$K_a\subseteq K_b\longrightarrow H_p(K_a;\mathbb F_2)\to H_p(K_b;\mathbb F_2)
\longrightarrow\text{persistence module}\longrightarrow\text{barcode}.$$

The hand calculations come first. A library calculation is used only after the spaces, maps and interval summary have been identified.

See the course **Applied glossary** for translations of *induced map*, *birth*, *death*, *persistence module*, *barcode* and *essential class*.

**Presenter route.** Use the hand calculation first. The software barcode is evidence that the same birth-and-death logic scales, not a replacement definition.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG=np.random.default_rng(3024)

def rank_mod2(A):
    A=np.array(A,dtype=np.uint8,copy=True)%2; row=rank=0
    for col in range(A.shape[1]):
        piv=np.flatnonzero(A[row:,col])
        if not len(piv): continue
        q=row+piv[0]; A[[row,q]]=A[[q,row]]
        for i in range(A.shape[0]):
            if i!=row and A[i,col]: A[i]^=A[row]
        row+=1; rank+=1
        if row==A.shape[0]: break
    return rank

def plot_barcode(diagrams, titles=('H0','H1')):
    fig,axes=plt.subplots(1,len(diagrams),figsize=(10,3.4))
    for dim,(ax,D) in enumerate(zip(np.atleast_1d(axes),diagrams)):
        finite=D[np.isfinite(D[:,1])] if len(D) else D
        cap=max(finite[:,1].max() if len(finite) else 1, D[:,0].max() if len(D) else 1)*1.08
        for y,(b,d) in enumerate(D): ax.hlines(y,b,cap if np.isinf(d) else d,lw=2.5)
        ax.set_title(titles[dim]); ax.set_xlabel('Rips distance threshold ε'); ax.set_yticks([]); ax.set_xlim(left=0)
    plt.tight_layout(); plt.show()

print('Using coefficients in F_2.')

## 1. Observe: lecture demonstration

**Presenter cue.** Show the object before the calculation. Ask the room to separate what is given from what will be constructed.

Let $X$ be the outline triangle and $Y$ the same vertices and edges plus the triangular face. The inclusion $X\hookrightarrow Y$ preserves every old simplex and chain.

The question is not whether the old edge cycle still exists as a chain. It does. The question is what happens to its class after $Y$ supplies a new 2-chain.

## 2. Predict: lecture demonstration

**Presenter cue.** Pause here and collect at least two predictions before revealing any output.

Before computing:

1. Predict $H_1(X;\mathbb F_2)$ and $H_1(Y;\mathbb F_2)$.
2. Is the induced map $H_1(X)\to H_1(Y)$ injective?
3. Two vertices merge after an edge is added. Which direction in $H_0\cong\mathbb F_2^2$ is killed?
4. If two unrelated classes live on $[1,4)$ and $[2,6)$, what is the dimension of the module at parameters 0, 1.5, 3 and 5?

## 3. Implement: lecture demonstration

**Reveal.** Run one cell at a time. Name the domain, codomain, complex, module or summary before interpreting its values.

### A. A class dies when it becomes a boundary

The cycle space of the three-edge outline is one-dimensional. Compare the image of $\partial_2$ before and after the face is added.

In [ ]:
d2_X=np.zeros((3,0),dtype=np.uint8)
d2_Y=np.ones((3,1),dtype=np.uint8)
dim_Z1=1
beta1_X=dim_Z1-rank_mod2(d2_X)
beta1_Y=dim_Z1-rank_mod2(d2_Y)
print('beta_1(X)=',beta1_X,'beta_1(Y)=',beta1_Y)

The same edge cycle spans $Z_1$ in both complexes. In $X$, $B_1=0$; in $Y$, the cycle spans $B_1=\operatorname{im}\partial_2$. Thus its class maps to zero.

### B. Components merge through a non-injective map

In bases $(e_1,e_2)$ before the edge and $(e)$ afterwards, the induced map is represented by $[1\ 1]$.

In [ ]:
H0_map=np.array([[1,1]],dtype=np.uint8)
for v in [np.array([1,0]),np.array([0,1]),np.array([1,1])]:
    print(v,'maps to',(H0_map@v)%2)

Both $e_1$ and $e_2$ map to $e$, while $e_1+e_2$ maps to zero over $\mathbb F_2$. The rank drops from two to one.

## 4. Compare: lecture demonstration

**Controlled comparison.** Keep the stated input fixed and change only the highlighted modelling decision.

### A. Betti counts versus module maps

Two modules can have the same dimension at every sampled parameter but connect their vector spaces differently. A sequence of Betti numbers records only vertical slice sizes. The persistence module retains the compatible maps.

For the two intervals $[1,4)$ and $[2,6)$, count how many bars cross each requested parameter.

In [ ]:
bars=[(1.,4.,'A'),(2.,6.,'B')]
fig,ax=plt.subplots(figsize=(8,2.8))
for y,(b,d,label) in enumerate(bars):
    ax.hlines(y,b,d,lw=5); ax.text(d+.12,y,label,va='center')
ax.set_xlim(0,7); ax.set_yticks([]); ax.set_xlabel('filtration parameter'); ax.set_title('Two interval summands'); plt.show()

def dimension_at(a): return sum(b<=a<d for b,d,_ in bars)
for a in [0,1.5,3,5]: print(a,dimension_at(a))

The dimensions at $0,1.5,3,5$ are respectively $0,1,2,1$. These are vertical slices of the barcode; they do not by themselves recover the cross-scale maps.

### Triangle filtration checkpoint: worked

For the order $v_0,v_1,e_{01},v_2,e_{12},e_{02},f_{012}$:

| Addition | Event |
|---|---|
| Each vertex | An $H_0$ class is born |
| $e_{01}$ and $e_{12}$ | The younger component class dies at each merge |
| $e_{02}$ | Its endpoints are already connected, so an $H_1$ class is born |
| $f_{012}$ | Its boundary is the three-edge cycle, so that $H_1$ class dies |

Matrix reduction pairs the two connecting edges with younger vertex births and pairs $f_{012}$ with $e_{02}$. The edge-cycle chain remains present after the face arrives, but it has become a boundary.

### B. From hand calculation to a library barcode

We now use `ripser` on two deterministic synthetic data sets: a noisy circle and a filled disk. `ripser` reports the Rips **pairwise-distance threshold** $\varepsilon$, so $\varepsilon=2r$ relative to Week 3's ball-radius convention.

Before running the cell, predict which data set should have the larger maximum finite $H_1$ persistence. Also inspect the radial coefficient of variation as a simpler baseline.

In [ ]:
n=90
theta=np.linspace(0,2*np.pi,n,endpoint=False)
circle=np.c_[np.cos(theta),np.sin(theta)]+0.035*RNG.normal(size=(n,2))
u=RNG.random(n); phi=2*np.pi*RNG.random(n)
disk=np.c_[np.sqrt(u)*np.cos(phi),np.sqrt(u)*np.sin(phi)]

fig,axes=plt.subplots(1,2,figsize=(7,3.2))
for ax,P,title in zip(axes,[circle,disk],['noisy circle','filled disk']):
    ax.scatter(P[:,0],P[:,1],s=14); ax.set_aspect('equal'); ax.set_title(title); ax.axis('off')
plt.show()

for name,P in [('circle',circle),('disk',disk)]:
    radii=np.linalg.norm(P-P.mean(axis=0),axis=1)
    print(name,'radial coefficient of variation =',round(radii.std()/radii.mean(),3))

In [ ]:
from ripser import ripser

circle_dgms=ripser(circle,maxdim=1)['dgms']
disk_dgms=ripser(disk,maxdim=1)['dgms']
print('Noisy circle barcode'); plot_barcode(circle_dgms)
print('Filled disk barcode'); plot_barcode(disk_dgms)

def longest_finite(D):
    finite=D[np.isfinite(D[:,1])]
    return float(np.max(finite[:,1]-finite[:,0])) if len(finite) else 0.0

for name,D in [('circle',circle_dgms[1]),('disk',disk_dgms[1])]:
    print(name,'longest finite H1 persistence =',round(longest_finite(D),3))

## 5. Interpret: lecture demonstration

**Presenter close.** Ask what the example supports, what information was discarded and which stronger claim would be unjustified.

1. Why can inclusion of complexes induce a non-injective map on homology?
2. What information do module maps retain that Betti numbers discard?
3. Under what one-parameter finiteness or tameness conditions is a barcode a complete interval summary?
4. What does an infinite death mean in this computed filtration, and when could it instead reflect truncation?
5. Did the topological calculation add anything beyond the radial baseline for this deliberately simple comparison?
6. Which Week 6 question begins once two diagrams need to be compared?

**◇ Object check.** The noisy samples, Rips filtration, persistence module, barcode and longest-bar scalar are different objects. Each arrow discards or adds information.

**† Qualification.** These are synthetic demonstrations, not empirical evidence about a dynamical system.

### Worked interpretation

The face added to $Y$ makes the old cycle a boundary, so its nonzero class maps to zero. In $H_0$, both component generators map to one later generator and $e_1+e_2$ lies in the kernel over $\mathbb F_2$. Module maps therefore retain survival and merging information that pointwise dimensions forget.

For finite filtrations of finite complexes over a field, the resulting one-parameter module admits the interval decomposition used here. More general statements require an appropriate tameness condition. An infinite death means unpaired within the computed filtration; a restricted maximum threshold can also censor a later death.

The radial baseline already separates this specially designed circle/disk example, so the barcode has not demonstrated unique practical value. Week 6 begins the next argument: how diagrams are compared, how perturbations are bounded and how a summary is validated.

## Lecture close

Return to the final slide questions.

1. Name the observed or starting object.
2. Name every constructed object used in this walkthrough.
3. Identify the single modelling decision that drove the central comparison.
4. State one conclusion supported by the calculation and one conclusion it cannot establish.

**Take-forward example.** The purpose of a small class that dies under inclusion, followed by a noisy-circle barcode is to make persistent homology as homology with memory concrete. The example is deliberately small or synthetic so that the construction remains inspectable.